# A second axis has a different alternative set

Are different classifier fields sufficient for two different alternative sets?

Run cells from top to bottom in a fresh Python kernel using the repository environment. Set the working directory to the repository root. This notebook is self-contained; do not run another declaration experiment in the same kernel. The matching `.py` file runs with `uv run python examples/step_21_relations/38_error_axis_subset.py`.

This is a deliberately broken declaration. The error is caught and printed; the corrected declaration is explained at the end. Read the [diagnostic guide](../../docs/how-to/specialization-declaration-fails.md) for the working baseline.


## Imports

These imports supply the annotation tools and AOA types used below.

In [1]:
from __future__ import annotations

from typing import Annotated, Literal

from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity, BaseSpecEntity, Classifier, Generalization, Inverse, Rel, Specialization
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


## MusicDomain

Group the music catalogue declarations. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [2]:
class MusicDomain(BaseDomain):
    """Group the music catalogue declarations."""

    name = "music"
    description = "A music catalogue"


## VinylRecordCoreEntity

Describe the information shared by all pressings. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [3]:
@entity(description="Vinyl record", domain=MusicDomain)
class VinylRecordCoreEntity(BaseCoreEntity):
    """Describe the information shared by all pressings."""

    id: str = Field(description="Record identifier")
    title: str = Field(description="Album title")
    media: str = Field(description="Pressing code")
    reported_media: str = Field(description="Pressing code reported by another catalogue")
    reported_pressing: Annotated[
        Specialization[FirstPressSpecEntity | RepressSpecEntity],
        Classifier(field="reported_media", codes=Literal["first", "repress"]),
        Inverse(field_name="record"),
    ] = Rel(description="Pressing reported by another catalogue")
    pressing: Annotated[
        Specialization[FirstPressSpecEntity | RepressSpecEntity | TestPressSpecEntity],
        Classifier(field="media", codes=Literal["first", "repress", "test"]),
        Inverse(field_name="record"),
    ] = Rel(description="Details of this pressing")


## FirstPressSpecEntity

Describe the stamper used for a first pressing. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [4]:
@entity(description="First pressing", domain=MusicDomain)
class FirstPressSpecEntity(BaseSpecEntity):
    """Describe the stamper used for a first pressing."""

    id: str = Field(description="The head record identifier")
    stamper: str = Field(description="Stamper code")
    record: Annotated[
        Generalization[VinylRecordCoreEntity],
        Classifier("record", Literal["first"]),
        Inverse(VinylRecordCoreEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## RepressSpecEntity

Describe when an album was pressed again. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [5]:
@entity(description="Re-pressing", domain=MusicDomain)
class RepressSpecEntity(BaseSpecEntity):
    """Describe when an album was pressed again."""

    id: str = Field(description="The head record identifier")
    year: int = Field(description="Re-pressing year")
    record: Annotated[
        Generalization[VinylRecordCoreEntity],
        Classifier("record", Literal["repress"]),
        Inverse(VinylRecordCoreEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## TestPressSpecEntity

Describe who approved a test pressing. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [6]:
@entity(description="Test pressing", domain=MusicDomain)
class TestPressSpecEntity(BaseSpecEntity):
    """Describe who approved a test pressing."""

    id: str = Field(description="The head record identifier")
    approved_by: str = Field(description="Reviewer name")
    record: Annotated[
        Generalization[VinylRecordCoreEntity],
        Classifier("record", Literal["test"]),
        Inverse(VinylRecordCoreEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## Resolve the declared types

Pydantic can now resolve the references between classes. This does not create record objects or load data.

In [7]:
for model in (VinylRecordCoreEntity, FirstPressSpecEntity, RepressSpecEntity, TestPressSpecEntity):
    model.model_rebuild()


## The experiment

Are different classifier fields sufficient for two different alternative sets?

In [8]:
from aoa.action_machine.domain.exceptions import SpecializationDeclarationError
from aoa.action_machine.graph.validators.entity_specialization_validator import validate_entity_specializations

try:
    validate_entity_specializations()
except SpecializationDeclarationError as error:
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("The broken declaration was accepted")


SpecializationDeclarationError: Specialization 'reported_pressing' on entity 'VinylRecordCoreEntity' is invalid: class 'TestPressSpecEntity' points at 'reported_pressing' without being named in the union — add it to the alternatives or remove its reverse field


## Why this result occurs

TestPressSpecEntity still points at VinylRecordCoreEntity. The current closure check compares that reverse link with every axis on the head, so it rejects TestPressSpecEntity as outside reported_pressing. Distinct classifier fields alone are not sufficient.

## Repair

Restore the shared alternative set, or use separate heads when the two choices need different sets. This example documents a current restriction; it does not implement a workaround in the framework.

Apply that change to the declaration above, restart the kernel, and run again. Replace the experiment cell with the following success check. This corrected model has also been run through an actual machine build:

```python
from aoa.action_machine.graph.validators.entity_specialization_validator import validate_entity_specializations
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

validate_entity_specializations()
ActionProductMachine(loggers=[])
print("Corrected model built")
```

Expected output:

```text
Corrected model built
```